# EMEP country-level analysis: PM10, POD1 and POD3

This notebook reads the EMEP five-year-average NetCDF file and the grid-to-country fraction NetCDF file, calculates **area-weighted country means**, and exports an Excel workbook.

**Important choices to review:**
- `SURF_ug_PM10` is used for PM10 surface concentration (µg/m³), not the alternative `SURF_ug_PM10_rh50` field.
- EMEP contains several POD1 and POD3 variants. The main three-output table defaults to `POD1_DF` and `POD3_TC`. Change these below to the vegetation/impact indicators appropriate for your analysis.
- Country fractions are used as percentages (0–1), multiplied by grid-cell area. This correctly handles grid cells that straddle borders.
- The source is already a five-year average of 2016–2020. Results represent that average; they are not five separate annual observations.

**Python version:** Python 3.11 or newer (tested with Python 3.11.15)
**Compiler:** JupyterLab or classic Jupyter Notebook  

**Dependencies:** see [`requirements.txt`](../requirements.txt) in the repository root (install with `pip install -r requirements.txt`).

In [2]:
# If needed, uncomment and run once to install NetCDF readers:
#%pip install xarray netCDF4 h5netcdf pandas openpyxl numpy

from pathlib import Path
import re
import numpy as np
import pandas as pd
import xarray as xr
import h5py, h5netcdf

print("xarray:", xr.__version__)


xarray: 2026.9.0


In [21]:
# Select the variables for the main three-output dataframe.
PM10_VAR = "SURF_ug_PM10"  # µg/m³; alternative: SURF_ug_PM10_rh50
POD1_VAR = "POD1_DF"        # e.g. POD1_CF, POD1_SNL, POD1_IAM_DF
POD3_VAR = "POD3_TC"        # e.g. POD3_IAM_WH, POD3_IAM_WH_MED, etc.

# ---------- CONFIG ----------
COUNTRY_FILE = Path(r"Raw_Data\EMEP_raw_data\emep_ll_gridfraction_01degCEIP_2018_ospar2020_eez.nc")
OUTPUT_XLSX_v7 = Path("Raw_Data/EMEP_country_PM10_POD1_POD3_v7.xlsx")
OUTPUT_XLSX_v5 = Path("Raw_Data/EMEP_country_PM10_POD1_POD3_v5.xlsx")

# One entry per file. Variable names differ between files, so they are set per file.
EMEP_FILES_v7 = [
    {"label": "Base_v7a", "Year": 2040, "path": Path(r"Raw_Data\EMEP_raw_data\v7aa_vcorr\EMEP01_annual_baseline2040_5year_average.nc"),
     "PM10_VAR": PM10_VAR, "POD1_VAR": POD1_VAR, "POD3_VAR": POD3_VAR},
    {"label": "MTFR_v7a", "Year": 2040, "path": Path(r"Raw_Data\EMEP_raw_data\v7aa_vcorr\EMEP01_annual_mtfr2040_5year_average.nc"), 
     "PM10_VAR": PM10_VAR, "POD1_VAR": POD1_VAR, "POD3_VAR": POD3_VAR},
    {"label": "Health 50%, Bio 50%-v7a", "Year": 2040, "path": Path(r"Raw_Data\EMEP_raw_data\v7aa_vcorr\EMEP01_annual_J5050_2040_5year_average.nc"), 
     "PM10_VAR": PM10_VAR, "POD1_VAR": POD1_VAR, "POD3_VAR": POD3_VAR},
    {"label": "Health 55%, Bio 45%-v7a", "Year": 2040, "path": Path(r"Raw_Data\EMEP_raw_data\v7aa_vcorr\EMEP01_annual_J5545_2040_5year_average.nc"), 
     "PM10_VAR": PM10_VAR, "POD1_VAR": POD1_VAR, "POD3_VAR": POD3_VAR},
    {"label": "Health 60%, Bio 40%-v7a", "Year": 2040, "path": Path(r"Raw_Data\EMEP_raw_data\v7aa_vcorr\EMEP01_annual_J6040_2040_5year_average.nc"), 
     "PM10_VAR": PM10_VAR, "POD1_VAR": POD1_VAR, "POD3_VAR": POD3_VAR},
    {"label": "Health 60%, Bio 45%-v7a", "Year": 2040, "path": Path(r"Raw_Data\EMEP_raw_data\v7aa_vcorr\EMEP01_annual_J6045_2040_5year_average.nc"), 
     "PM10_VAR": PM10_VAR, "POD1_VAR": POD1_VAR, "POD3_VAR": POD3_VAR},
    {"label": "Base_v7a", "Year": 2050, "path": Path(r"Raw_Data\EMEP_raw_data\v7aa_vcorr\EMEP01_annual_baseline2050_5year_average.nc"),
     "PM10_VAR": PM10_VAR, "POD1_VAR": POD1_VAR, "POD3_VAR": POD3_VAR},
    {"label": "MTFR_v7a", "Year": 2050, "path": Path(r"Raw_Data\EMEP_raw_data\v7aa_vcorr\EMEP01_annual_mtfr2050_5year_average.nc"), 
     "PM10_VAR": PM10_VAR, "POD1_VAR": POD1_VAR, "POD3_VAR": POD3_VAR},
    {"label": "Health 50%, Bio 50%-v7a", "Year": 2050, "path": Path(r"Raw_Data\EMEP_raw_data\v7aa_vcorr\EMEP01_annual_J5050_2050_5year_average.nc"), 
     "PM10_VAR": PM10_VAR, "POD1_VAR": POD1_VAR, "POD3_VAR": POD3_VAR},
    {"label": "Health 55%, Bio 45%-v7a", "Year": 2050, "path": Path(r"Raw_Data\EMEP_raw_data\v7aa_vcorr\EMEP01_annual_J5545_2050_5year_average.nc"), 
     "PM10_VAR": PM10_VAR, "POD1_VAR": POD1_VAR, "POD3_VAR": POD3_VAR},
    {"label": "Health 60%, Bio 40%-v7a", "Year": 2050, "path": Path(r"Raw_Data\EMEP_raw_data\v7aa_vcorr\EMEP01_annual_J6040_2050_5year_average.nc"), 
     "PM10_VAR": PM10_VAR, "POD1_VAR": POD1_VAR, "POD3_VAR": POD3_VAR},
    {"label": "Health 60%, Bio 45%-v7a", "Year": 2050, "path": Path(r"Raw_Data\EMEP_raw_data\v7aa_vcorr\EMEP01_annual_J6045_2050_5year_average.nc"), 
     "PM10_VAR": PM10_VAR, "POD1_VAR": POD1_VAR, "POD3_VAR": POD3_VAR}
]

EMEP_FILES_v5 = [
    {"label": "CLE 2040", "path": Path(r"Raw_Data\EMEP_raw_data\v5\EMEP01_annual.cle2040_5year_average.nc"), 
     "PM10_VAR": PM10_VAR, "POD1_VAR": POD1_VAR, "POD3_VAR": POD3_VAR},
    {"label": "MTFR", "path": Path(r"Raw_Data\EMEP_raw_data\v5\EMEP01_annual.mtfr2040_5year_average.nc"), 
     "PM10_VAR": PM10_VAR, "POD1_VAR": POD1_VAR, "POD3_VAR": POD3_VAR},
    {"label": "OPT_HV 2040", "path": Path(r"Raw_Data\EMEP_raw_data\v5\EMEP01_annual.opt_hv2040_5year_average.nc"), 
     "PM10_VAR": PM10_VAR, "POD1_VAR": POD1_VAR, "POD3_VAR": POD3_VAR},
    {"label": "OPT 2040", "path": Path(r"Raw_Data\EMEP_raw_data\v5\EMEP01_annual.opt2040_5year_average.nc"), 
     "PM10_VAR": PM10_VAR, "POD1_VAR": POD1_VAR, "POD3_VAR": POD3_VAR}
]

mapping_countries = {
    "AL": "Albania",
    "AM": "Armenia",
    "AT": "Austria",
    "AZ": "Azerbaijan",
    "BA": "Bosnia-H",
    "BE": "Belgium",
    "BG": "Bulgaria",
    "BY": "Belarus",
    "CH": "Switzerland",
    "CY": "Cyprus",
    "CZ": "Czech Rep.",
    "DE": "Germany",
    "DK": "Denmark",
    "EE": "Estonia",
    "ES": "Spain",
    "FI": "Finland",
    "FR": "France",
    "GB": "United Kingdom",
    "GE": "Georgia",
    "GL": "Greenland",
    "GR": "Greece",
    "HR": "Croatia",
    "HU": "Hungary",
    "IE": "Ireland",
    "IS": "Iceland",
    "IT": "Italy",
    "KG": "Kyrgyzstan",
    "KZ": "Kazakhstan",
    "LI": "Liechtenstein",
    "LT": "Lithuania",
    "LU": "Luxembourg",
    "LV": "Latvia",
    "MC": "Monaco",
    "MD": "R Moldova",
    "ME": "Montenegro",
    "MK": "North Macedonia",
    "MT": "Malta",
    "NL": "Netherlands",
    "NO": "Norway",
    "PL": "Poland",
    "PT": "Portugal",
    "RO": "Romania",
    "RS": "Serbia",
    "RU": "Russia",
    "SE": "Sweden",
    "SI": "Slovenia",
    "SK": "Slovakia",
    "TJ": "Tajikistan",
    "TM": "Turkmenistan",
    "TR": "Türkiye",
    "UA": "Ukraine",
    "UZ": "Uzbekistan"
}

In [4]:
# ---------- HELPER FUNCTIONS ----------
def to_2d(da, name):
    """float64 numpy array, shape (lat, lon), fill values -> NaN."""
    for dim in [d for d in da.dims if d not in ("lat", "lon")]:
        if da.sizes[dim] != 1:
            raise ValueError(f"{name} has {da.sizes[dim]} entries along {dim!r}; expected 1.")
        da = da.isel({dim: 0})
    arr = da.transpose("lat", "lon").values.astype("float64")
    for key in ("_FillValue", "missing_value"):
        fill = da.attrs.get(key, da.encoding.get(key))
        if fill is not None:
            arr[arr == float(np.ravel(fill)[0])] = np.nan
    arr[~np.isfinite(arr) | (np.abs(arr) >= 1e20)] = np.nan
    return arr

def weighted_mean(field, weights):
    valid = np.isfinite(field)
    w = np.where(valid, weights, 0.0)
    w_sum = w.sum()
    if w_sum <= 0:
        return np.nan, 0.0
    mean = np.sum(np.where(valid, field, 0.0) * w) / w_sum
    return mean, w_sum / weights.sum()

def find_var(ds, explicit, prefix):
    if explicit is not None:
        if explicit not in ds.data_vars:
            raise KeyError(f"{explicit!r} not found. Available: {[v for v in ds.data_vars if v.startswith(prefix)]}")
        return explicit
    matches = [v for v in ds.data_vars if v.startswith(prefix)]
    if len(matches) != 1:
        raise ValueError(f"Expected exactly one variable starting with {prefix!r}, found {matches}. Set it explicitly in EMEP_FILES.")
    return matches[0]

def get_attr_text(value):
    if isinstance(value, bytes):
        return value.decode("utf-8", errors="replace")
    if isinstance(value, np.ndarray) and value.size == 1:
        return get_attr_text(value.item())
    return str(value) if value is not None else ""



In [5]:
# ---------- ONE-TIME SETUP: country grid ----------

if not COUNTRY_FILE.exists():
    raise FileNotFoundError(COUNTRY_FILE.resolve())

with xr.open_dataset(COUNTRY_FILE, engine="h5netcdf") as grid:
    grid = grid.load()          # small enough; closes the file handle afterwards

country_layers = []
for name, da in grid.data_vars.items():
    units = get_attr_text(da.attrs.get("units", "")).strip()
    code = get_attr_text(da.attrs.get("long_name", "")).strip().upper()
    if units == "%" and re.fullmatch(r"[A-Z]{2}", code) and da.ndim == 2:
        country_layers.append((name, code))
country_layers.sort(key=lambda x: x[1])
if not country_layers:
    raise ValueError("No country fraction layers found.")

cell_area = to_2d(grid["cell_area"], "cell_area")
cell_area = np.where(np.isfinite(cell_area) & (cell_area > 0), cell_area, 0.0)

# Pre-compute weights per country once (reused for every EMEP file)
country_weights = {}
for layer_name, code in country_layers:
    frac = to_2d(grid[layer_name], layer_name)
    frac = np.where(np.isfinite(frac) & (frac > 0), frac, 0.0)
    country_weights[code] = cell_area * frac

grid_lat, grid_lon = grid["lat"].values, grid["lon"].values

## Calculate area-weighted country averages

For each country, the weight for each grid cell is `cell area × country fraction`. The fraction is converted from percent to a 0–1 share. Concentration and POD indicators are spatially averaged, not summed. This produces a representative country mean for each field.

In [12]:
# ---------- PER-FILE PROCESSING ----------
def process_emep_file(cfg):
    path = cfg["path"]
    if not path.exists():
        raise FileNotFoundError(path.resolve())

    with xr.open_dataset(path, engine="h5netcdf") as emep:
        # Coordinate check against the country grid
        if (emep.sizes["lat"] != grid_lat.size or emep.sizes["lon"] != grid_lon.size
                or not np.allclose(emep["lat"].values, grid_lat)
                or not np.allclose(emep["lon"].values, grid_lon)):
            raise ValueError(f"{path.name}: lat/lon do not match the country grid.")

        pm10 = find_var(emep, cfg.get("PM10_VAR"), "PM10")
        pod1 = find_var(emep, cfg.get("POD1_VAR"), "POD1")
        pod3 = find_var(emep, cfg.get("POD3_VAR"), "POD3")
        fields = {
            "PM10_surface_ug_m3": to_2d(emep[pm10], pm10),
            "POD1": to_2d(emep[pod1], pod1),
            "POD3": to_2d(emep[pod3], pod3),
        }

    rows = []
    for code, weights in country_weights.items():
        row = {"Scenario": cfg["label"], "country_code": code, "Year":cfg["Year"]}
        for out_name, field in fields.items():
            row[out_name], row[f"{out_name}_coverage"] = weighted_mean(field, weights)
        row["PM10_variable"], row["POD1_variable"], row["POD3_variable"] = pm10, pod1, pod3
        rows.append(row)
    return pd.DataFrame(rows)

In [14]:
# ---------- RUN ALL v7aa FILES ----------
frames = []
for cfg in EMEP_FILES_v7:
    print("Processing", cfg["label"])
    frames.append(process_emep_file(cfg))

all_df = pd.concat(frames, ignore_index=True)
all_df["Country"] = all_df["country_code"].replace(mapping_countries)

value_cols = ["PM10_surface_ug_m3", "POD1", "POD3"]
# Fail loudly if a whole column is NaN for any file
nan_check = all_df.groupby("Scenario")[value_cols].apply(lambda d: d.isna().all())
assert not nan_check.any().any(), f"All-NaN column(s) for some file:\n{nan_check}"

export_cols = ["Scenario", "country_code", "Country", "Year", *value_cols,
               "PM10_variable", "POD1_variable", "POD3_variable"]
result_v7 = all_df[export_cols].sort_values(["Scenario", "country_code", "Year"]).reset_index(drop=True)


Processing Base_v7a
Processing MTFR_v7a
Processing Health 50%, Bio 50%-v7a
Processing Health 55%, Bio 45%-v7a
Processing Health 60%, Bio 40%-v7a
Processing Health 60%, Bio 45%-v7a
Processing Base_v7a
Processing MTFR_v7a
Processing Health 50%, Bio 50%-v7a
Processing Health 55%, Bio 45%-v7a
Processing Health 60%, Bio 40%-v7a
Processing Health 60%, Bio 45%-v7a


In [16]:
OUTPUT_XLSX_v7.parent.mkdir(parents=True, exist_ok=True)
result_v7.to_excel(OUTPUT_XLSX_v7, index=False)


In [17]:
# ---------- RUN ALL v5 FILES ----------
frames = []
for cfg in EMEP_FILES_v5:
    print("Processing", cfg["label"])
    frames.append(process_emep_file(cfg))

all_df = pd.concat(frames, ignore_index=True)
all_df["Country"] = all_df["country_code"].replace(mapping_countries)

value_cols = ["PM10_surface_ug_m3", "POD1", "POD3"]
# Fail loudly if a whole column is NaN for any file
nan_check = all_df.groupby("Scenario")[value_cols].apply(lambda d: d.isna().all())
assert not nan_check.any().any(), f"All-NaN column(s) for some file:\n{nan_check}"

export_cols = ["Scenario", "country_code", "Country", *value_cols,
               "PM10_variable", "POD1_variable", "POD3_variable"]
result_v5 = all_df[export_cols].sort_values(["Scenario", "country_code"]).reset_index(drop=True)

Processing CLE 2040
Processing MTFR
Processing OPT_HV 2040
Processing OPT 2040


In [19]:
OUTPUT_XLSX_v5.parent.mkdir(parents=True, exist_ok=True)
result_v5.to_excel(OUTPUT_XLSX_v5, index=False)
